# Runtime Classification

This notebook runs the complete OTSU-GWWHO framework on a single new MRI image: pre-processing, OTSU-GWWHO ROI segmentation, VGG16-DCNN classification into MD, MOD, ND or VMD, and an optional Grad-CAM explanation. The time taken by each stage is reported.

## Prerequisites

- Python 3.10+
- Required libraries: `numpy`, `opencv-python`, `matplotlib`, `tensorflow`
- A trained model from `VGG16_DCNN_classification.ipynb`

## Workflow

1. Load and pre-process the input MRI image (grayscale, 224×224, intensity normalisation)
2. Segment the ROI with OTSU-GWWHO
3. Classify the ROI with the trained VGG16-DCNN
4. Display the predicted class, class probabilities and processing time
5. Generate the Grad-CAM explanation

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input

IMAGE_PATH = "./sample_mri.jpg"
MODEL_PATH = "./outputs/06_models/vgg16_dcnn_fold0.keras"
CLASS_NAMES = ["MD", "MOD", "ND", "VMD"]
IMG_SIZE = 224
LAST_CONV_LAYER = "block5_conv3"
RANDOM_SEED = 42

POP_SIZE = 30        # N
MAX_ITER = 50        # T_max
PC = 0.13            # crossover (mating) probability
PS = 0.20            # stallion proportion
A_MIN = 0.01
A_MAX = 0.01
SIGMA_G = 32.0       # Gaussian parameter sigma_G
T_MIN, T_MAX = 0, 255
GRAY_LEVELS = 256

model = tf.keras.models.load_model(MODEL_PATH)

## Pre-processing

Same pre-processing as `image_preprocessing.ipynb`.

In [ ]:
# ==========================================================
# PRE-PROCESSING
# ==========================================================

def preprocess_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return np.round(img * 255).astype(np.uint8)

## OTSU-GWWHO Segmentation

Same OTSU-GWWHO algorithm as `OTSU_GWWHO_segmentation.ipynb`.

In [ ]:
# ==========================================================
# OTSU-GWWHO SEGMENTATION
# ==========================================================

def normalized_histogram(img):
    """Eq. (3): d(i) = n_i / (R*C)"""
    hist = np.bincount(img.ravel(), minlength=GRAY_LEVELS).astype(np.float64)
    return hist / hist.sum()


def within_class_variance_table(d):
    """Eq. (4)-(6): sigma_w^2(t) = w_b*sigma_b^2 + w_f*sigma_f^2 for every threshold t."""
    i = np.arange(d.size, dtype=np.float64)
    w_b = np.cumsum(d)
    m_b = np.cumsum(i * d)
    s_b = np.cumsum(i * i * d)
    w_f = 1.0 - w_b
    m_f = m_b[-1] - m_b
    s_f = s_b[-1] - s_b
    with np.errstate(divide="ignore", invalid="ignore"):
        var_b = s_b / w_b - (m_b / w_b) ** 2
        var_f = s_f / w_f - (m_f / w_f) ** 2
        sigma_w = w_b * var_b + w_f * var_f
    sigma_w[(w_b <= 0) | (w_f <= 0)] = np.inf
    return np.nan_to_num(sigma_w, nan=np.inf)


def otsu_fitness(table, t):
    idx = np.clip(np.rint(t), 0, table.size - 1).astype(int)
    return table[idx]


def gaussian_weight(dist):
    """Eq. (12): GW = exp(-d^2 / (2*sigma_G^2))"""
    return np.exp(-(dist ** 2) / (2.0 * SIGMA_G ** 2))


def GWWHO_threshold(img, seed=None, plot_curve=False):
    rng = np.random.default_rng(seed)
    table = within_class_variance_table(normalized_histogram(img))
    fit = lambda x: otsu_fitness(table, x)
    clip = lambda x: np.clip(x, T_MIN, T_MAX)

    # Population initialisation, Eq. (8)
    pos = T_MIN + rng.random(POP_SIZE) * (T_MAX - T_MIN)
    cost = fit(pos)

    # Groups and stallions, Eq. (9): G = N x PS
    n_groups = max(1, int(round(POP_SIZE * PS)))
    order = np.argsort(cost)
    stallion, stallion_cost = pos[order[:n_groups]].copy(), cost[order[:n_groups]].copy()
    foals = pos[order[n_groups:]].copy()
    foal_group = np.arange(foals.size) % n_groups
    rng.shuffle(foal_group)
    foal_cost = fit(foals)

    # Initial water-hole (best threshold), Eq. (7)
    wp, wp_cost = stallion[np.argmin(stallion_cost)], stallion_cost.min()
    best_fit_history, best_T_history = [], []

    for t in range(MAX_ITER):
        # Adaptive search coefficient, Eq. (10)
        A_t = A_MIN + (1 - t / MAX_ITER) * (A_MAX - A_MIN)

        # Grazing / mating, Eq. (11)-(15)
        for i in range(foals.size):
            g = foal_group[i]
            if rng.random() > PC or n_groups == 1:
                d = abs(foals[i] - stallion[g])                                    # Eq. (11)
                GW = gaussian_weight(d)                                            # Eq. (12)
                R = rng.uniform(-2, 2)
                new = GW * (2 * A_t * np.cos(2 * np.pi * R * A_t)
                            * (stallion[g] - foals[i])) + stallion[g]              # Eq. (13)
            else:
                others = np.flatnonzero(foal_group != g)
                j = rng.choice(others)
                new = (foals[i] + foals[j]) / 2.0                                  # Eq. (14)
            foals[i] = clip(new)
            foal_cost[i] = fit(foals[i])

        # Gaussian-weighted leadership and leader exchange, Eq. (16)-(18)
        for g in range(n_groups):
            R = rng.uniform(-2, 2)
            GW = gaussian_weight(abs(wp - stallion[g]))
            step = 2 * A_t * np.cos(2 * np.pi * R * A_t) * GW * (wp - stallion[g])
            cand = clip(step + wp if rng.random() > 0.5 else step - wp)            # Eq. (16)
            c_cost = fit(cand)
            if c_cost < stallion_cost[g]:
                stallion[g], stallion_cost[g] = cand, c_cost
            members = np.flatnonzero(foal_group == g)
            if members.size:
                b = members[np.argmin(foal_cost[members])]
                if foal_cost[b] < stallion_cost[g]:                                # Eq. (18)
                    foals[b], stallion[g] = stallion[g], foals[b]
                    foal_cost[b], stallion_cost[g] = stallion_cost[g], foal_cost[b]

        # Update water-hole (global best), Eq. (17)
        g_best = np.argmin(stallion_cost)
        if stallion_cost[g_best] < wp_cost:
            wp, wp_cost = stallion[g_best], stallion_cost[g_best]
        best_fit_history.append(float(wp_cost))
        best_T_history.append(int(np.rint(wp)))

    # Final optimal threshold, Eq. (19)
    T_star = int(np.clip(np.rint(wp), T_MIN, T_MAX))

    if plot_curve:
        plt.figure(figsize=(9, 4))
        plt.subplot(1, 2, 1)
        plt.plot(best_fit_history, linewidth=3)
        plt.title("GWWHO Convergence (Fitness)")
        plt.xlabel("Iteration"); plt.ylabel("Within-class variance")
        plt.subplot(1, 2, 2)
        plt.plot(best_T_history, linewidth=3)
        plt.title("Threshold Evolution")
        plt.xlabel("Iteration"); plt.ylabel("Threshold")
        plt.tight_layout(); plt.show()

    return T_star, float(table[T_star])


def GWWHO_segment(img, seed=None, plot_curve=False):
    """Eq. (20): I_s(r,c) = 1 if I(r,c) > T* else 0. Returns T*, mask and ROI."""
    if img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    T_star, fitness = GWWHO_threshold(img, seed, plot_curve)
    mask = (img > T_star).astype(np.uint8)
    roi = img * mask
    return T_star, fitness, mask, roi

## Classification and Grad-CAM

Classifies the segmented ROI and computes the Grad-CAM localisation map.

In [ ]:
# ==========================================================
# CLASSIFY & EXPLAIN
# ==========================================================

def to_model_input(roi):
    x = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB).astype(np.float32)
    return preprocess_input(x)[None, ...]


def grad_cam(x, class_idx):
    grad_model = tf.keras.Model(model.inputs, [model.get_layer(LAST_CONV_LAYER).output, model.output])
    with tf.GradientTape() as tape:
        fmap, preds = grad_model(x, training=False)
        y_c = preds[:, class_idx]
    alpha = tf.reduce_mean(tape.gradient(y_c, fmap), axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(fmap * alpha[:, None, None, :], axis=-1))[0].numpy()
    return cv2.resize(cam / (cam.max() + 1e-8), (IMG_SIZE, IMG_SIZE))

## Run the Pipeline

Processes the input image and reports the predicted Alzheimer's stage.

In [ ]:
# ==========================================================
# RUN
# ==========================================================

t0 = time.perf_counter()
img = preprocess_image(IMAGE_PATH)
t1 = time.perf_counter()
T_star, fitness, mask, roi = GWWHO_segment(img, seed=RANDOM_SEED)
t2 = time.perf_counter()
x = to_model_input(roi)
probs = model.predict(x, verbose=0)[0]
pred = int(np.argmax(probs))
t3 = time.perf_counter()

print(f"OTSU-GWWHO threshold T* : {T_star}")
print(f"Predicted class         : {CLASS_NAMES[pred]}")
for name, p in zip(CLASS_NAMES, probs):
    print(f"   {name:<4} {p * 100:6.2f} %")
print(f"Time (ms): pre-processing {1e3 * (t1 - t0):.1f} | segmentation {1e3 * (t2 - t1):.1f} | "
      f"classification {1e3 * (t3 - t2):.1f} | total {1e3 * (t3 - t0):.1f}")

cam = grad_cam(tf.convert_to_tensor(x), pred)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("Input MRI")
ax[1].imshow(mask, cmap="gray"); ax[1].set_title(f"Mask (T* = {T_star})")
ax[2].imshow(roi, cmap="gray"); ax[2].set_title("Segmented ROI")
ax[3].imshow(roi, cmap="gray"); ax[3].imshow(cam, cmap="jet", alpha=0.4)
ax[3].set_title(f"Grad-CAM: {CLASS_NAMES[pred]}")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()